In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, col, count, sum as _sum,
    min as _min, max as _max, round as _round
)

category_stats = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)

print("=== Statystyki per kategoria ===")
category_stats.show()

=== Statystyki per kategoria ===
+-----------+---------+----------+-------+-------+
|   category|liczba_tx|  suma_PLN|min_PLN|max_PLN|
+-----------+---------+----------+-------+-------+
|elektronika|     2542|1520770.69|    9.0| 9999.0|
|    książki|     2574| 851382.08|    5.0|9107.25|
|     odzież|     2453| 849877.55|    5.0|9696.63|
|    żywność|     2431| 789514.43|    5.0|6916.92|
+-----------+---------+----------+-------+-------+



In [10]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [11]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, col, count, sum as _sum,
    round as _round, window
)

half_hourly_store = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od", "store")
)

print("=== Transakcje per sklep w oknach 30-minutowych ===")
half_hourly_store.show(truncate=False)

=== Transakcje per sklep w oknach 30-minutowych ===
+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, col, count, sum as _sum,
    round as _round, window, desc
)

krakow_best_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy(desc("suma_PLN"))
)

print("=== Kraków — godzina z najwyższym przychodem ===")
krakow_best_hour.show(truncate=False)

=== Kraków — godzina z najwyższym przychodem ===
+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169     |483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|821      |341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|532      |201259.26|
+-------------------+-------------------+---------+---------+



In [16]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [19]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Sliding ma ~2x więcej wierszy bo okna się nakładają.
# Przy tumbling 3 godziny = 3 okna.
# Przy sliding 1h/30min = każda godzina pokryta przez 2 okna,
# więc całkowita liczba okien jest ~2x większa.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
#############################Praca domowa############################

In [20]:
gdansk_min_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "srednia_PLN",
    )
    .orderBy("srednia_PLN")
)

print("=== Gdańsk — godzina z najniższą średnią kwotą ===")
gdansk_min_avg.show(truncate=False)

=== Gdańsk — godzina z najniższą średnią kwotą ===
+-------------------+-------------------+---------+-----------+
|od                 |do                 |liczba_tx|srednia_PLN|
+-------------------+-------------------+---------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|766      |395.01     |
|2026-04-12 10:00:00|2026-04-12 11:00:00|558      |412.92     |
|2026-04-12 09:00:00|2026-04-12 10:00:00|1174     |415.91     |
+-------------------+-------------------+---------+-----------+



In [ ]:
W gdańsku, najniższe średnie kwoty transakcji występują od 8:00 do 9:00

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, col, count, sum as _sum,
    min as _min, max as _max, round as _round,
    to_date, lit
)

first_date = df.select(
    to_date(col("timestamp")).alias("data")
).first()["data"]

print(f"Data w danych: {first_date}")

start_time = f"{first_date} 09:00:00"
end_time   = f"{first_date} 09:30:00"

window_09_0930 = (
    df.filter(
        (col("timestamp") >= lit(start_time).cast("timestamp")) &
        (col("timestamp") <  lit(end_time).cast("timestamp"))
    )
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)

print("=== Transakcje per kategoria w oknie 09:00–09:30 ===")
window_09_0930.show(truncate=False)

Data w danych: 2026-04-12
=== Transakcje per kategoria w oknie 09:00–09:30 ===
+-----------+---------+---------+-------+-------+
|category   |liczba_tx|suma_PLN |min_PLN|max_PLN|
+-----------+---------+---------+-------+-------+
|elektronika|611      |349852.93|9.0    |9999.0 |
|książki    |622      |191895.44|5.0    |2637.18|
|odzież     |605      |204888.51|5.0    |6013.37|
|żywność    |567      |175645.23|7.12   |2704.16|
+-----------+---------+---------+-------+-------+



In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, col, count, sum as _sum,
    round as _round, window, desc
)

peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy(desc("liczba_tx"))
)

print("=== Szczyt transakcji — okna 15-minutowe ===")
peak_15min.show(20, truncate=False)

print("=== Absolutny szczyt (1 wiersz) ===")
peak_15min.show(1, truncate=False)

=== Szczyt transakcji — okna 15-minutowe ===
+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |481566.97|
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |440715.14|
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |504943.74|
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |475251.18|
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |469004.36|
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |355500.31|
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |359254.89|
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |213061.19|
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |224438.4 |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |198098.62|
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |156900.51|
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |132809.44|
+-------------------+----

In [24]:
spark.stop()